### FAISS
FAISS stands for Facebook AI Simillar Search is a library for efficient simillar search and clustering of dense vectors. It contains algorithm that search in sets of vectors of any size, upto ones that possibly do not fit in RAM. It also contains supporting code for evalution and parameter tuning.

Note: Install "faiss-cpu" for FAISS

In [8]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
## from langchain_community.embeddings import OllamaEmbeddings
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

## Load Data
loader = TextLoader("speech.txt")
documents = loader.load()

## Split Text into chunks
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=30)
docs = text_splitter.split_documents(documents)

## Ollama Ebeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")

## Get text chunks and embed each chunks using Ollama Embeddings 
## and store those chunks+their vectors in Chroma vector Database
db = FAISS.from_documents(docs, embeddings)


[Document(id='de43c3f4-15fc-4860-b400-e080a239d7cc', metadata={'source': 'speech.txt'}, page_content="The Art of Building with AI\nGood morning, everyone. Today I want to talk about the incredible journey of building intelligent systems that can understand, reason, and assist us in our daily work.\nArtificial intelligence has evolved from simple rule-based systems to sophisticated language models capable of understanding context, generating coherent text, and even reasoning through complex problems. This transformation didn't happen overnight. It is the result of decades of research in machine learning, neural networks, and natural language processing.\nWhen we think about frameworks like LangChain, we are really talking about the plumbing that connects large language models to the real world. It allows developers to chain together prompts, models, tools, and data sources to build applications that go far beyond a simple chatbot. From document question answering to autonomous agents, t

"The Art of Building with AI\nGood morning, everyone. Today I want to talk about the incredible journey of building intelligent systems that can understand, reason, and assist us in our daily work.\nArtificial intelligence has evolved from simple rule-based systems to sophisticated language models capable of understanding context, generating coherent text, and even reasoning through complex problems. This transformation didn't happen overnight. It is the result of decades of research in machine learning, neural networks, and natural language processing.\nWhen we think about frameworks like LangChain, we are really talking about the plumbing that connects large language models to the real world. It allows developers to chain together prompts, models, tools, and data sources to build applications that go far beyond a simple chatbot. From document question answering to autonomous agents, the possibilities are expanding every single day.\nData ingestion is often the unsung hero of any AI p

In [9]:
## Querying post Storing Chunks in Vector DB
query = "What is Data ingestion ?"
result = db.similarity_search(query)
print(result)
result[0].page_content

[Document(id='de43c3f4-15fc-4860-b400-e080a239d7cc', metadata={'source': 'speech.txt'}, page_content="The Art of Building with AI\nGood morning, everyone. Today I want to talk about the incredible journey of building intelligent systems that can understand, reason, and assist us in our daily work.\nArtificial intelligence has evolved from simple rule-based systems to sophisticated language models capable of understanding context, generating coherent text, and even reasoning through complex problems. This transformation didn't happen overnight. It is the result of decades of research in machine learning, neural networks, and natural language processing.\nWhen we think about frameworks like LangChain, we are really talking about the plumbing that connects large language models to the real world. It allows developers to chain together prompts, models, tools, and data sources to build applications that go far beyond a simple chatbot. From document question answering to autonomous agents, t

"The Art of Building with AI\nGood morning, everyone. Today I want to talk about the incredible journey of building intelligent systems that can understand, reason, and assist us in our daily work.\nArtificial intelligence has evolved from simple rule-based systems to sophisticated language models capable of understanding context, generating coherent text, and even reasoning through complex problems. This transformation didn't happen overnight. It is the result of decades of research in machine learning, neural networks, and natural language processing.\nWhen we think about frameworks like LangChain, we are really talking about the plumbing that connects large language models to the real world. It allows developers to chain together prompts, models, tools, and data sources to build applications that go far beyond a simple chatbot. From document question answering to autonomous agents, the possibilities are expanding every single day.\nData ingestion is often the unsung hero of any AI p

### As a Retriever
We can also convert the vectorstore into a Retiever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers.

In [11]:
retriever = db.as_retriever()
retriever.invoke(query)
docs[0].page_content

"The Art of Building with AI\nGood morning, everyone. Today I want to talk about the incredible journey of building intelligent systems that can understand, reason, and assist us in our daily work.\nArtificial intelligence has evolved from simple rule-based systems to sophisticated language models capable of understanding context, generating coherent text, and even reasoning through complex problems. This transformation didn't happen overnight. It is the result of decades of research in machine learning, neural networks, and natural language processing.\nWhen we think about frameworks like LangChain, we are really talking about the plumbing that connects large language models to the real world. It allows developers to chain together prompts, models, tools, and data sources to build applications that go far beyond a simple chatbot. From document question answering to autonomous agents, the possibilities are expanding every single day.\nData ingestion is often the unsung hero of any AI p

### Simillarity Search with score
There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. THerefore, a lower score is better.

In [12]:
docs_and_score_db = db.similarity_search_with_score(query)
docs_and_score_db

[(Document(id='de43c3f4-15fc-4860-b400-e080a239d7cc', metadata={'source': 'speech.txt'}, page_content="The Art of Building with AI\nGood morning, everyone. Today I want to talk about the incredible journey of building intelligent systems that can understand, reason, and assist us in our daily work.\nArtificial intelligence has evolved from simple rule-based systems to sophisticated language models capable of understanding context, generating coherent text, and even reasoning through complex problems. This transformation didn't happen overnight. It is the result of decades of research in machine learning, neural networks, and natural language processing.\nWhen we think about frameworks like LangChain, we are really talking about the plumbing that connects large language models to the real world. It allows developers to chain together prompts, models, tools, and data sources to build applications that go far beyond a simple chatbot. From document question answering to autonomous agents, 

In [13]:
### Save in Local and Loading

db.save_local("faiss_index") ## faiss_index is the index name we have given

In [18]:
new_db = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
new_db_result = new_db.similarity_search(query)
new_db_result

[Document(id='de43c3f4-15fc-4860-b400-e080a239d7cc', metadata={'source': 'speech.txt'}, page_content="The Art of Building with AI\nGood morning, everyone. Today I want to talk about the incredible journey of building intelligent systems that can understand, reason, and assist us in our daily work.\nArtificial intelligence has evolved from simple rule-based systems to sophisticated language models capable of understanding context, generating coherent text, and even reasoning through complex problems. This transformation didn't happen overnight. It is the result of decades of research in machine learning, neural networks, and natural language processing.\nWhen we think about frameworks like LangChain, we are really talking about the plumbing that connects large language models to the real world. It allows developers to chain together prompts, models, tools, and data sources to build applications that go far beyond a simple chatbot. From document question answering to autonomous agents, t